<a href="https://colab.research.google.com/github/Elementos-Reconocimiento-Visual-UBA/PracticasERV_2026_4toBim/blob/main/Tarea_1_MIN_CUT_ST_y_NCUT/Tarea_1_Parte_2_%7C_Aplicaciones_de_Video_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Nombre de Grupos*:

*Integrantes*:

# Parte 2: Segmentación de Video (Espacio-Temporal)

Hasta ahora segmentamos imágenes estáticas (2D) en la primera parte.No obstante, hacerlo en video es solo tener una variable más (volumen de pixeles apilados en el tiempo). Por eso ahora pasamos de coordenadas $(x, y)$ a coordenadas **$(x, y, t)$**.

Para segmentar este volumen sin que el resultado "parpadee" ruidosamente de un frame a otro, necesitamos actualizar nuestros grafos. Veamos qué cambia exactamente en los dos métodos que estudiamos:

---

### 1. Cambios en la topología general (Los N-links)
En ambos algoritmos (MinCut y Ncut), los nodos ya no se conectan solo con sus vecinos espaciales inmediatos.

* **En 2D:** Un píxel se conectaba con sus vecinos arriba, abajo, izquierda y derecha dentro de la misma imagen (en MinCut) o con sus vecinos dado un radio espacial (en Ncut)
* **En Video:** Ahora, el píxel $(x, y)$ en el frame $t$ también se conecta con los píxeles en el **frame siguiente** $(t+1)$ y anterior $(t-1)$. Es decir, se va a conectar con sus vecinos espaciales de su frame y tambien con los vecinos espaciales de los otros 2 frames (si antes miraba 4 ahora pasa a mirar 4 más en el futuro y 4 más del pasado)

Además para manejar estas neuvas aristas "temporales" se introduce $\lambda_t$
* Si $\lambda_t = 0$, borramos las conexiones en el tiempo. Segmentar el video es equivalente a procesar cada frame por separado (genera parpadeo).
* Si $\lambda_t$ es equilibrado, los frames se "atan" entre sí, logrando que el segmento mantenga una **coherencia temporal** mientras el objeto se mueve.

---

### 2. Cambios en Corte Mínimo s-t (Aparición de las GMM)
En la parte de imágenes, los enlaces a la Fuente (Fondo) y al Sumidero (Objeto) —los **T-links**— se definían manualmente: donde el usuario pintaba, el costo era infinito, y en el resto era cero.

En un video, el objeto se mueve y cambia de forma. ¡Es imposible pedirle al usuario que pinte el objeto frame por frame! Para resolver esto, usamos un enfoque estadístico:

1. **Semillas iniciales:** El usuario pinta un par de trazos en *un solo frame*.
2. **Aprendizaje de color (GMM):** El algoritmo toma todos los píxeles bajo el trazo del "objeto" y ajusta un Modelo de Mezcla Gaussiana (GMM). Hace lo mismo con el trazo del "fondo". Una GMM aprende la distribución de colores 3D (por ejemplo, aprende que el objeto es mayormente rojo y azul, y el fondo es verde).
3. **T-links probabilísticos:** En los frames donde el usuario *no* pintó, el algoritmo evalúa cada píxel y calcula: *"¿Qué probabilidad hay de que este color pertenezca a la GMM del objeto?"*. Esa probabilidad estadística se convierte automáticamente en el peso del T-link para ese píxel.

---

### 3. Cambios en Normalized Cuts
La lógica matemática de Ncut (resolver autovectores) se mantiene intacta, pero ocurren dos cambios a nivel computacional:

1. **La matriz $W$ es 3D:** La matriz de afinidad ahora pesa distancias de color y distancias espacio-temporales (o sea, tiene que ver espacio, espacio temporal y color)
2. **Partición sin recursión:** Como el grafo 3D es gigantesco, la búsqueda recursiva de umbrales que hicimos en la Parte 1 se vuelve muy lenta. En su lugar, el paper original propone usar los primeros $k$ autovectores para proyectar los píxeles en un espacio continuo, hacer un agrupamiento rápido con *K-Means* (sobresegmentación), y luego fusionar ("podar") los pedacitos adyacentes que minimicen la fórmula del Ncut multiclase.

## 1. Herramientas

No hace falta leer esta celda: tiene la carga del video, los dos métodos en video y funciones para mostrar y guardar.

In [ ]:
#@title Herramientas: carga de video, corte mínimo s-t y Ncut (no hace falta leerlas) { display-mode: "form" }
import os
import sys
import cv2
import uuid
import json
import time
import base64
import numpy as np
import imageio.v2 as imageio
from scipy import ndimage
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, JSON, display
from sklearn.mixture import GaussianMixture
import itertools
import scipy.sparse as sp
import scipy.sparse.linalg as sla
from sklearn.cluster import KMeans

#Instalación dependencias
def _instalar(modulo, paquete):
    import subprocess
    try:
        return __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', paquete], check=True)
        return __import__(modulo)

plt.rcParams['animation.embed_limit'] = 60

VIDEOS_DE_EJEMPLO = {
    'webcam': 'https://raw.githubusercontent.com/opencv/opencv_extra/4.x/testdata/cv/tracking/faceocc2/data/faceocc2.webm',
    'peatones': 'https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/vtest.avi',
    'animacion': 'https://raw.githubusercontent.com/opencv/opencv_extra/4.x/testdata/highgui/video/big_buck_bunny.mp4',
}

def obtener_video(ruta):
    if ruta in VIDEOS_DE_EJEMPLO:
        url = VIDEOS_DE_EJEMPLO[ruta]
        local = os.path.basename(url)
        if not os.path.exists(local):
            import urllib.request
            urllib.request.urlretrieve(url, local)
        return local
    if ruta is None:
        from google.colab import files
        return next(iter(files.upload()))
    return ruta

def info_video(ruta):
    cap = cv2.VideoCapture(ruta)
    fps = cap.get(cv2.CAP_PROP_FPS)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    ancho, alto = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return fps, n, ancho, alto

def mostrar_recorte(ruta, t, recorte):
    cap = cv2.VideoCapture(ruta)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(t * cap.get(cv2.CAP_PROP_FPS)))
    ok, f = cap.read(); cap.release()
    x0, y0, w, h = recorte
    plt.figure(figsize=(7, 4.5))
    plt.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    plt.gca().add_patch(plt.Rectangle((x0, y0), w, h, fill=False, color='red', lw=2))
    plt.title(f'Frame en t = {t} s (rojo = recorte)'); plt.show()

def agrandar(x, forma):
    '''Lleva máscaras (T, H, W) bool o etiquetas (T, H, W) int a otra resolución (H2, W2), por vecino más cercano.'''
    T = x.shape[0]
    alto, ancho = forma
    tipo = x.dtype
    salida = np.stack([cv2.resize(x[t].astype(np.int32 if tipo != bool else np.uint8), (ancho, alto),
                                  interpolation=cv2.INTER_NEAREST) for t in range(T)])
    return salida.astype(tipo)

def limpiar_mascaras(mascaras, rellenar=True):
    '''En cada frame se queda con la componente conexa más grande de la máscara (saca manchitas sueltas)'''
    salida = np.zeros_like(mascaras, dtype=bool)
    for t, m in enumerate(mascaras):
        etiquetas, n = ndimage.label(m)
        if n == 0: continue
        tam = ndimage.sum(m, etiquetas, range(1, n + 1))
        mayor = etiquetas == (1 + int(np.argmax(tam)))
        salida[t] = ndimage.binary_fill_holes(mayor) if rellenar else mayor
    return salida

def cargar_clip(ruta, t0, t1, n, lado, recorte=None, lado_hd=360):
    cap = cv2.VideoCapture(ruta)
    fps = cap.get(cv2.CAP_PROP_FPS)
    ancho, alto = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    x0, y0, w, h = recorte if recorte is not None else (0, 0, ancho, alto)
    escala = lambda m: (max(1, round(w * m / max(w, h))), max(1, round(h * m / max(w, h))))
    i0, i1 = int(t0 * fps), int(t1 * fps)
    elegidos = set(np.linspace(i0, i1 - 1, n).round().astype(int))
    cap.set(cv2.CAP_PROP_POS_FRAMES, i0)
    chicos, grandes = [], []
    for i in range(i0, i1):
        ok, f = cap.read()
        if not ok: break
        if i in elegidos:
            f = cv2.cvtColor(f[y0:y0 + h, x0:x0 + w], cv2.COLOR_BGR2RGB)
            chicos.append(cv2.resize(f, escala(lado), interpolation=cv2.INTER_AREA))
            grandes.append(cv2.resize(f, escala(lado_hd), interpolation=cv2.INTER_AREA))
    cap.release()
    if not chicos: raise ValueError('No se leyó ningún frame. Revisá T_INICIO y T_FIN.')
    fps_clip = len(chicos) / (t1 - t0)
    return (np.stack(chicos).astype(np.float64) / 255, np.stack(grandes).astype(np.float32) / 255, fps_clip)

def mostrar(paneles, titulos, fps=6):
    T = paneles[0].shape[0]
    fig, axs = plt.subplots(1, len(paneles), figsize=(3.6 * len(paneles), 3.8))
    axs = np.atleast_1d(axs)
    ims = []
    for ax, p, ti in zip(axs, paneles, titulos):
        ims.append(ax.imshow(np.clip(p[0], 0, 1), cmap='gray', vmin=0, vmax=1))
        ax.set_title(ti); ax.axis('off')
    def actualizar(t):
        for im, p in zip(ims, paneles): im.set_data(np.clip(p[t], 0, 1))
        fig.suptitle(f'frame {t + 1}/{T}')
        return ims
    anim = animation.FuncAnimation(fig, actualizar, frames=T, interval=1000 / fps)
    plt.close(fig)
    return HTML(anim.to_jshtml())

def a_rgb(x):
    x = np.asarray(x, dtype=np.float32)
    return np.repeat(x[..., None], 3, -1) if x.ndim == 3 else x

def lado_a_lado(*clips, alto=None):
    clips = [a_rgb(c) for c in clips]
    alto = alto or clips[0].shape[1]
    salida = []
    for c in clips:
        ancho = round(c.shape[2] * alto / c.shape[1])
        salida.append(np.stack([cv2.resize(f, (ancho, alto), interpolation=cv2.INTER_NEAREST) for f in c]))
    return np.concatenate(salida, axis=2)

def guardar_video(clip, nombre, fps=6, descargar=True):
    frames = (255 * np.clip(a_rgb(clip), 0, 1)).astype(np.uint8)
    if nombre.endswith('.gif'):
        imageio.mimsave(nombre, frames, duration=1000 / fps, loop=0)
    else:
        frames = frames[:, :frames.shape[1] // 2 * 2, :frames.shape[2] // 2 * 2] # mp4 pide lados pares
        imageio.mimsave(nombre, frames, fps=fps, macro_block_size=1)
    if descargar:
        try:
            from google.colab import files
            files.download(nombre)
        except ImportError: pass
#para UI
def _a_png(imagen_rgba):
    ok, buf = cv2.imencode('.png', cv2.cvtColor(imagen_rgba, cv2.COLOR_RGBA2BGRA))
    return 'data:image/png;base64,' + base64.b64encode(buf).decode()

def _capa_mascara(objeto_t):
    capa = np.zeros(objeto_t.shape + (4,), np.uint8)
    capa[~objeto_t] = (0, 0, 0, 150)
    borde = objeto_t & ~cv2.erode(objeto_t.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)
    capa[borde] = (255, 220, 0, 255)
    return capa

RESULTADO_ST = {'mascaras': None, 'semillas': None, 'trazos': None}

INTERFAZ_ST = '''
<div id="UID" style="font-family: sans-serif; font-size: 14px;">
  <div style="display: flex; gap: 16px; align-items: flex-start; flex-wrap: wrap;">
    <div>
      <canvas class="lienzo" style="border: 1px solid #888; cursor: crosshair; touch-action: none;"></canvas>
      <div style="margin-top: 6px;">
        Frame <input class="frame" type="range" min="0" value="0" style="width: 260px; vertical-align: middle;">
        <span class="frame-lbl"></span>
      </div>
    </div>
    <div style="display: flex; flex-direction: column; gap: 8px; min-width: 200px;">
      <label><input type="radio" name="et-UID" value="3" checked> <b style="color:#2e7d32">Pintar encima del objeto</b></label>
      <label><input type="radio" name="et-UID" value="1"> <b style="color:#e53935">Corregir: objeto (t)</b></label>
      <label><input type="radio" name="et-UID" value="2"> <b style="color:#1e88e5">Corregir: fondo (s)</b></label>
      <label>Pincel <input class="radio" type="range" min="1" max="15" value="6" style="vertical-align: middle;"></label>
      <label><input class="ver" type="checkbox" checked> Ver segmentación</label>
      <button class="seg" style="padding: 6px; font-weight: bold;">Segmentar</button>
      <button class="play">Reproducir</button>
      <button class="undo">Deshacer último trazo</button>
      <button class="clear">Borrar todos los trazos</button>
      <div class="msg" style="max-width: 220px; color: #555;"></div>
    </div>
  </div>
</div>
<script>
(function () {
  const raiz = document.getElementById('UID');
  const FRAMES = __FRAMES__;
  const T = FRAMES.length, H = __H__, W = __W__, ZOOM = __ZOOM__;
  const COLORES = {1: 'rgba(229,57,53,0.85)', 2: 'rgba(30,136,229,0.85)'};
  const lienzo = raiz.querySelector('.lienzo'), ctx = lienzo.getContext('2d');
  lienzo.width = W * ZOOM; lienzo.height = H * ZOOM;
  const capa = document.createElement('canvas'), cctx = capa.getContext('2d');
  capa.width = lienzo.width; capa.height = lienzo.height;
  const sliderFrame = raiz.querySelector('.frame'), lblFrame = raiz.querySelector('.frame-lbl');
  const sliderRadio = raiz.querySelector('.radio'), chkVer = raiz.querySelector('.ver');
  const msg = raiz.querySelector('.msg'), btnPlay = raiz.querySelector('.play');
  sliderFrame.max = T - 1;

  const imgs = FRAMES.map(src => { const im = new Image(); im.onload = dibujar; im.src = src; return im; });
  let mascaras = null, trazos = [], actual = null, temporizador = null;

  function frame() { return parseInt(sliderFrame.value); }
  function etiqueta() { return parseInt(raiz.querySelector('input[name="et-UID"]:checked').value); }

  function dibujar() {
    const t = frame();
    lblFrame.textContent = (t + 1) + '/' + T;
    ctx.imageSmoothingEnabled = false;
    ctx.clearRect(0, 0, lienzo.width, lienzo.height);
    if (imgs[t].complete) ctx.drawImage(imgs[t], 0, 0, lienzo.width, lienzo.height);
    if (mascaras && chkVer.checked && mascaras[t].complete) ctx.drawImage(mascaras[t], 0, 0, lienzo.width, lienzo.height);
    cctx.clearRect(0, 0, capa.width, capa.height);
    const todos = trazos.concat(actual ? [actual] : []);
    for (const tr of todos) {
      for (const [ft, x, y, r, e] of tr) {
        if (ft !== t || e !== 3) continue;
        cctx.fillStyle = 'rgb(67,160,71)';
        cctx.beginPath(); cctx.arc(x * ZOOM, y * ZOOM, Math.max(r, 0.5) * ZOOM, 0, 2 * Math.PI); cctx.fill();
      }
    }
    ctx.globalAlpha = (mascaras && chkVer.checked) ? 0.2 : 0.4;
    ctx.drawImage(capa, 0, 0);
    ctx.globalAlpha = 1;
    for (const tr of todos) {
      for (const [ft, x, y, r, e] of tr) {
        if (ft !== t || e === 3) continue;
        ctx.fillStyle = COLORES[e];
        ctx.beginPath(); ctx.arc(x * ZOOM, y * ZOOM, Math.max(r, 0.5) * ZOOM, 0, 2 * Math.PI); ctx.fill();
      }
    }
  }

  function posicion(ev) {
    const rect = lienzo.getBoundingClientRect();
    return [(ev.clientX - rect.left) * (lienzo.width / rect.width) / ZOOM,
            (ev.clientY - rect.top) * (lienzo.height / rect.height) / ZOOM];
  }

  function agregarPunto(ev) {
    const [x, y] = posicion(ev), r = parseInt(sliderRadio.value), t = frame(), e = etiqueta();
    const ultimo = actual[actual.length - 1];
    if (ultimo) {
      const dx = x - ultimo[1], dy = y - ultimo[2], pasos = Math.ceil(Math.hypot(dx, dy) / Math.max(r / 2, 0.5));
      for (let k = 1; k < pasos; k++) actual.push([t, ultimo[1] + dx * k / pasos, ultimo[2] + dy * k / pasos, r, e]);
    }
    actual.push([t, x, y, r, e]);
    dibujar();
  }

  lienzo.addEventListener('pointerdown', ev => { actual = []; lienzo.setPointerCapture(ev.pointerId); agregarPunto(ev); });
  lienzo.addEventListener('pointermove', ev => { if (actual) agregarPunto(ev); });
  const terminar = () => { if (actual && actual.length) trazos.push(actual); actual = null; dibujar(); };
  lienzo.addEventListener('pointerup', terminar);
  lienzo.addEventListener('pointercancel', terminar);

  sliderFrame.addEventListener('input', dibujar);
  chkVer.addEventListener('change', dibujar);
  raiz.querySelector('.undo').addEventListener('click', () => { trazos.pop(); dibujar(); });
  raiz.querySelector('.clear').addEventListener('click', () => { trazos = []; mascaras = null; msg.textContent = ''; dibujar(); });

  btnPlay.addEventListener('click', () => {
    if (temporizador) { clearInterval(temporizador); temporizador = null; btnPlay.textContent = 'Reproducir'; return; }
    btnPlay.textContent = 'Pausar';
    temporizador = setInterval(() => { sliderFrame.value = (frame() + 1) % T; dibujar(); }, 170);
  });

  raiz.querySelector('.seg').addEventListener('click', async () => {
    if (!(window.google && google.colab && google.colab.kernel)) { msg.textContent = 'Esto funciona solo en Google Colab.'; return; }
    msg.textContent = 'Calculando...';
    try {
      const puntos = [].concat(...trazos);
      const res = await google.colab.kernel.invokeFunction('notebook.mincut_st', [puntos], {});
      const datos = res.data['application/json'];
      if (datos.error) { msg.textContent = datos.error; return; }
      mascaras = datos.mascaras.map(src => { const im = new Image(); im.onload = dibujar; im.src = src; return im; });
      chkVer.checked = true;
      msg.textContent = datos.info + ' ¡Listo!';
    } catch (err) { msg.textContent = 'Error: ' + err; }
  });

  dibujar();
})();
</script>
'''

def interfaz_st(clip, params, zoom=4):
    T, H, W, _ = clip.shape
    def _segmentar(puntos):
        semillas = rasterizar_trazos(puntos, (T, H, W))
        if not (semillas == 3).any() and not ((semillas == 1).any() and (semillas == 2).any()):
            return JSON({'error': 'Pintá por encima del objeto (o marcá al menos una corrección de objeto y una de fondo).'})
        t0 = time.time()
        mascaras, _ = mincut_st(clip, semillas, **params)
        RESULTADO_ST.update(mascaras=mascaras, semillas=semillas, trazos=puntos)
        capas = [_a_png(_capa_mascara(mascaras[t])) for t in range(T)]
        return JSON({'mascaras': capas, 'info': f'Listo en {time.time()-t0:.1f} s.'})
    try:
        from google.colab import output
        output.register_callback('notebook.mincut_st', _segmentar)
    except ImportError: return

    frames_png = [_a_png(np.dstack([(255*f).astype(np.uint8), np.full(f.shape[:2], 255, np.uint8)])) for f in clip]
    uid = 'st' + uuid.uuid4().hex[:8]
    display(HTML(INTERFAZ_ST.replace('UID', uid).replace('__FRAMES__', json.dumps(frames_png))
                 .replace('__H__', str(H)).replace('__W__', str(W)).replace('__ZOOM__', str(zoom))))


#Mincut s-t
maxflow = _instalar('maxflow', 'PyMaxflow')

def rasterizar_trazos(trazos, forma):
    semillas = np.zeros(forma, np.uint8)
    for t, x, y, r, e in trazos:
        cv2.circle(semillas[int(t)], (int(round(x)), int(round(y))), max(int(round(r)), 0), int(e), -1)
    return semillas

def preparar_semillas(semillas, margen=2):
    s = semillas.copy()
    nucleo = np.ones((2*margen + 1, 2*margen + 1), np.uint8)
    for t in range(len(s)):
        region = s[t] == 3
        if region.any():
            cerca = cv2.dilate(region.astype(np.uint8), nucleo).astype(bool)
            s[t][~cerca & (s[t] == 0)] = 2
    return s

def costo_color(clip, obj, fondo, n_comp=5, max_muestras=20000):
    F = clip.reshape(-1, 3)
    rng = np.random.default_rng(0)
    costos = []
    for m in (obj.ravel(), fondo.ravel()):
        X = F[m]
        if len(X) > max_muestras:
            X = X[rng.choice(len(X), max_muestras, replace=False)]
        gmm = GaussianMixture(min(n_comp, max(1, len(X) // 20)), covariance_type='full', reg_covar=1e-4, random_state=0).fit(X)
        costos.append(-gmm.score_samples(F))
    minimo = min(c.min() for c in costos)
    return [(c - minimo).reshape(clip.shape[:3]) for c in costos]

def aristas_vecinos(clip, gamma, r, rt, lam_t):
    T, H, W, _ = clip.shape
    difs = [((clip[:, :, 1:] - clip[:, :, :-1]) ** 2).sum(-1).ravel(),
            ((clip[:, 1:] - clip[:, :-1]) ** 2).sum(-1).ravel()]
    beta = 1 / (2 * np.concatenate(difs).mean() + 1e-12)
    lista = []
    for dt in range(0, rt + 1):
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if (dt, dy, dx) <= (0, 0, 0) or (dt > 0 and lam_t == 0): continue
                pesos = np.zeros((T, H, W))
                a = (slice(0, T - dt), slice(max(0, -dy), H - max(0, dy)), slice(max(0, -dx), W - max(0, dx)))
                b = (slice(dt, T), slice(max(0, dy), H + min(0, dy)), slice(max(0, dx), W + min(0, dx)))
                d2 = ((clip[a] - clip[b]) ** 2).sum(-1)
                dist = np.sqrt(dx * dx + dy * dy) if dt == 0 else 1.0
                pesos[a] = gamma * np.exp(-beta * d2) / dist * (lam_t if dt > 0 else 1.0)
                estructura = np.zeros((3, 3, 3))
                estructura[1 + dt, 1 + dy, 1 + dx] = 1
                lista.append((pesos, estructura))
    return lista

def mincut_st(clip, semillas, lam_color=1.0, gamma=50.0, r=1, rt=1, lam_t=1.0, iteraciones=3, margen=2):
    s = preparar_semillas(semillas, margen)
    if not ((s == 1) | (s == 3)).any() or not (s == 2).any():
        raise ValueError('Hacen falta trazos de objeto (o pintar encima del objeto) y de fondo.')
    vecinos = aristas_vecinos(clip, gamma, r, rt, lam_t)
    obj, fondo = (s == 1) | (s == 3), s == 2
    INF = 1e9
    mascaras, flujo = None, 0.0
    for _ in range(iteraciones):
        c_obj, c_fondo = costo_color(clip, obj, fondo)
        g = maxflow.Graph[float]()
        ids = g.add_grid_nodes(clip.shape[:3])
        for pesos, estructura in vecinos:
            g.add_grid_edges(ids, weights=pesos, structure=estructura, symmetric=True)
        cap_s = lam_color * c_obj
        cap_t = lam_color * c_fondo
        cap_s[s == 2], cap_t[s == 2] = INF, 0
        cap_s[s == 1], cap_t[s == 1] = 0, INF
        g.add_grid_tedges(ids, cap_s, cap_t)
        flujo = g.maxflow()
        nuevo = g.get_grid_segments(ids)
        if nuevo.sum() < 20 or (~nuevo).sum() < 20: break
        mascaras = nuevo
        obj, fondo = mascaras, ~mascaras
    if mascaras is None: mascaras = nuevo
    return mascaras, flujo

def linea(t, p0, p1, radio, etiqueta, n=30):
    '''Un trazo recto en el frame t, de p0 = (x0, y0) a p1 = (x1, y1), en coordenadas del clip.'''
    return [[t, p0[0] + (p1[0] - p0[0]) * k / n, p0[1] + (p1[1] - p0[1]) * k / n, radio, etiqueta] for k in range(n + 1)]

#Ncut
def construir_W(clip, r=2, rt=1, sigma_I=0.1, sigma_X=4.0, lam_t=1.0):
    T, H, W, C = clip.shape
    N = T * H * W
    F = clip.reshape(N, C)
    idx = np.arange(N).reshape(T, H, W)
    filas, cols, pesos = [], [], []
    for dt in range(0, rt + 1):
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if (dt, dy, dx) <= (0, 0, 0) or (dt > 0 and lam_t == 0): continue
                a = idx[0:T - dt, max(0, -dy):H - max(0, dy), max(0, -dx):W - max(0, dx)].ravel()
                b = idx[dt:T,     max(0, dy):H + min(0, dy),  max(0, dx):W + min(0, dx)].ravel()
                w = np.exp(-((F[a] - F[b]) ** 2).sum(1) / sigma_I ** 2) * np.exp(-(dx * dx + dy * dy) / sigma_X ** 2) * (lam_t if dt > 0 else 1.0)
                filas += [a, b]; cols += [b, a]; pesos += [w, w]
    return sp.coo_matrix((np.concatenate(pesos), (np.concatenate(filas), np.concatenate(cols))), shape=(N, N)).tocsr()

def ncut_autovectores(W, k=10, tol=1e-3):
    d = np.asarray(W.sum(1)).ravel() + 1e-12
    s = 1 / np.sqrt(d)
    M = (sp.diags(s) @ W @ sp.diags(s)).tocsr()
    v0 = np.random.default_rng(0).standard_normal(W.shape[0])
    mu, Z = sla.eigsh(M, k=k + 1, which='LA', tol=tol, ncv=max(40, 2*k + 3), v0=v0)
    orden = np.argsort(-mu)
    return 1 - mu[orden][1:], (Z[:, orden] * s[:, None])[:, 1:], d

def sobresegmentar(Y, k0=12, semilla=0):
    E = Y / (np.linalg.norm(Y, axis=1, keepdims=True) + 1e-12)
    return KMeans(k0, n_init=4, random_state=semilla).fit_predict(E)

def grafo_condensado(W, etiquetas, k0):
    n = len(etiquetas)
    P = sp.csr_matrix((np.ones(n), (np.arange(n), etiquetas)), shape=(n, k0))
    return (P.T @ W @ P).toarray()

def ncut_k(Wc, grupos):
    total = Wc.sum(1)
    valor = 0.0
    for gr in grupos:
        assoc = total[gr].sum()
        valor += (assoc - Wc[np.ix_(gr, gr)].sum()) / assoc
    return valor

def poda_greedy(Wc, k_final):
    grupos = [[i] for i in range(len(Wc))]
    historial = []
    while len(grupos) > k_final:
        mejor = None
        for a, b in itertools.combinations(range(len(grupos)), 2):
            propuesta = [g for i, g in enumerate(grupos) if i not in (a, b)] + [grupos[a] + grupos[b]]
            v = ncut_k(Wc, propuesta)
            if mejor is None or v < mejor[0]: mejor = (v, propuesta, grupos[a], grupos[b])
        v, grupos, ga, gb = mejor
        historial.append((ga, gb, v))
    return grupos, historial

def ncut_video(clip, params, k_autovectores=10, k0=12, k_final=3, verbose=True):
    T, H, W_, _ = clip.shape
    t0 = time.time()
    Wm = construir_W(clip, **params)
    t1 = time.time()
    lam, Y, d = ncut_autovectores(Wm, k=k_autovectores)
    t2 = time.time()
    et0 = sobresegmentar(Y, k0)
    grupos, historial = poda_greedy(grafo_condensado(Wm, et0, k0), k_final)
    mapa = np.empty(k0, int)
    for j, gr in enumerate(grupos): mapa[gr] = j
    t3 = time.time()
    if verbose:
        print(f'W: {Wm.shape[0]:,} nodos, {Wm.nnz / Wm.shape[0]:.0f} vecinos/nodo ({t1-t0:.1f} s)')
        print(f'Autovectores: {t2-t1:.1f} s. Sobresegmentación y poda: {t3-t2:.1f} s')
    forma = (T, H, W_)
    return {'etiquetas': mapa[et0].reshape(forma), 'sobreseg': et0.reshape(forma),
            'fiedler': Y[:, 0].reshape(forma), 'lam': lam, 'historial': historial,
            'tiempos': (t1 - t0, t2 - t1, t3 - t2)}

def tabla_segmentos(clip, etiquetas):
    F = clip.reshape(-1, 3); e = etiquetas.ravel()
    filas = []
    for c in np.unique(e):
        m = e == c
        filas.append({'segmento': int(c), 'tam': m.mean(), 'color': F[m].mean(0),
                      'contraste': float(np.linalg.norm(F[m].mean(0) - F[~m].mean(0)))})
    return filas

def pintar_segmentos(clip, etiquetas, alfa=0.6):
    paleta = plt.cm.tab10(np.arange(10))[:, :3]
    return (1 - alfa) * clip + alfa * paleta[etiquetas % 10]

def elegir_por_contraste(clip, etiquetas, tam_min=0.03):
    validos = [f for f in tabla_segmentos(clip, etiquetas) if f['tam'] >= tam_min]
    return max(validos, key=lambda f: f['contraste'])['segmento']



## 2. Carga y Preparación del Video

A diferencia de una imagen estática, procesar un video de alta resolución frame por frame haría que la matriz $W$ sea gigantesca e inmanejable para la RAM de Colab.
Por eso, no procesamos el video entero. Extraemos un **Clip**: un fragmento de pocos segundos, recortando solo el área de interés y bajándole la resolución.

- **`VIDEO`**: Podés usar uno de los ejemplos (`'webcam'`, `'peatones'`, `'animacion'`) o poner el nombre de un archivo tuyo (`'mi_video.mp4'`)

| Video de Ejemplo | Qué tiene |
|---|---|
| `'webcam'` | Una persona moviéndose (está en escala de grises) |
| `'peatones'` | Cámara fija, una persona caminando por el pasto |
| `'animacion'` | Un personaje grande en el centro. |

In [ ]:
VIDEO = '1ff8e768-6136-4137-ad8b-dab8a1f03e37.mp4'  #Opciones: 'webcam', 'peatones', 'animacion', 'mi_video.mp4'

EJEMPLOS = {
    'webcam': {'T_INICIO': 0, 'T_FIN': 3, 'RECORTE': None, 'N_FRAMES': 24},
    'peatones': {'T_INICIO': 59, 'T_FIN': 65, 'RECORTE': (0, 230, 460, 346), 'N_FRAMES': 24},
    'animacion': {'T_INICIO': 0, 'T_FIN': 5, 'RECORTE': None, 'N_FRAMES': 30},
}

# Si el video está en los ejemplos, usamos sus parámetros. Si es tuyo, usamos estos:
if VIDEO in EJEMPLOS:
    config = EJEMPLOS[VIDEO]
else:
    config = {
        'T_INICIO': 0,      #Segundo de inicio
        'T_FIN': 2,         #Segundo de fin
        'RECORTE': None,    #Ej: (x0, y0, ancho, alto) en px, o None para el cuadro completo
        'N_FRAMES': 24      #Cuántos frames muestrear en ese intervalo de tiempo
    }

#FIJAMOS EL LADO MÁS GRANDE Y LO HACEMOS PROPORCIONAL AL VIDEO ORIGINAL
LADO = 100

#Carga del video
RUTA = obtener_video(VIDEO)
fps_orig, n_total, w_orig, h_orig = info_video(RUTA)
print(f"Video original: {RUTA} | Resolución: {w_orig}x{h_orig} px | {fps_orig:.1f} FPS | Duración: {n_total/fps_orig:.1f} s")

#Mostramos el primer frame del fragmento elegido
mostrar_recorte(RUTA, config['T_INICIO'], config['RECORTE'] or (0, 0, w_orig, h_orig))

#Extraemos los clips
clip, clip_hd, fps_clip = cargar_clip(
    ruta=RUTA,
    t0=config['T_INICIO'],
    t1=config['T_FIN'],
    n=config['N_FRAMES'],
    lado=LADO,
    recorte=config['RECORTE']
)

T, H, W, _ = clip.shape
N_NODOS = T * H * W

print("-" * 50)
print(f"CLIP EXTRAÍDO (Baja resolución para Segmentar):")
print(f" Dimensiones: {T} frames x {W} ancho x {H} alto")
print(f" Grafo 3D: Tendrá un total de {N_NODOS:,} nodos.")
print(f" FPS final:{fps_clip:.1f} fps")

#Reproducimos la versión que el algoritmo va a "ver" realmente
mostrar([clip], ['Clip a segmentar (Baja Resolución)'], fps=fps_clip)

## 3.1 Segmentación: Corte Mínimo s-t (con trazos)

### ¿Cómo usar la interfaz?
1.  **Pintar encima del objeto (verde):** Elegí un frame donde se vea bien el objeto. Pintá con el pincel grueso por encima de **todo** el objeto (no pasa nada si te salís un poquito). El algoritmo asumirá que todo lo que esté lejos de ese trazo verde es "fondo seguro".
2.  **Tocar "Segmentar":** El laplaciano calculará el min-cut 3D.
3.  **Corregir objeto/fondo (rojo/azul):** Si en el frame 5 el algoritmo se equivocó y pintó pasto como si fuera persona, andá a ese frame, seleccioná "Corregir: fondo", pintá sobre el pasto y volvé a tocar Segmentar.

| Parámetro | Qué controla | Valor Inicial |
|---|---|---|
| `lam_color` | Peso del Modelo de Color. Si el objeto segmentado "desaparece" o tiene huecos, subilo. | 5.0 |
| `gamma` | Peso de la continuidad (vecinos). Más alto = bordes más suaves y redondeados, pero podés perder extremidades finas. | 10.0 |
| `lam_t` | Conexión temporal. Si lo ponés en 0, cada frame se corta solo (parpadea). | 1.0 |
| `iteraciones` | Vueltas de "cortar y reajustar GMM" (método GrabCut iterativo). | 3 |

## 3. Segmentar

Corré **solo el método que usa tu aplicación**: 3.A para 4.1, 4.2 y 4.3; 3.B para 4.4.

### 3.A Corte mínimo s-t (con trazos)

Con la convención del Laboratorio 2: la **fuente $s$ es el fondo** y el **sumidero $t$ es el objeto**. Además de las
aristas entre vecinos (y entre frames), cada píxel tiene un costo de color: con lo que pintaste se ajusta un modelo de
color del objeto y otro del fondo, como en GrabCut.

- **Pintar encima del objeto** (verde): pintá grueso por encima de **todo** el objeto en un frame, aunque te pases un
  poco. Lo lejano a lo pintado pasa a ser fondo seguro.
- **Corregir: objeto / fondo**: en cualquier frame donde se equivocó, y volvé a tocar **Segmentar**.

`MODO_ST = 'interfaz'` abre el pincel (solo en Colab). `MODO_ST = 'trazos'` usa trazos escritos como líneas: para los
videos de ejemplo ya vienen hechos.

| Parámetro | Qué controla | Inicial |
|---|---|---|
| `lam_color` | Cuánto pesa el modelo de color frente a los bordes. Subilo si el objeto "desaparece" | 5 |
| `gamma` | Peso de las aristas entre vecinos. Más alto = bordes más suaves, pero se pierden objetos chicos | 10 |
| `lam_t` | Peso de las aristas entre frames. Con 0, cada frame se corta por separado | 1 |
| `iteraciones` | Vueltas de "cortar y reajustar colores" | 3 |

In [ ]:
#Podés tunear estos parámetros y volver a ejecutar la celda
PARAMS_ST = dict(lam_color=5.0, gamma=10.0, r=1, rt=1, lam_t=1.0, iteraciones=3, margen=2)

#MODO = 'interfaz' abre el widget interactivo
#MODO = 'trazos' usa coordenadas programadas a mano (por si necesitas automatizar)
MODO_ST = 'interfaz'

# Trazos de ejemplo pre-programados (frame, (x_inicio, y_inicio), (x_fin, y_fin), grosor, etiqueta)
# Etiquetas: 3 = pintar encima (verde), 1 = corregir objeto (rojo), 2 = corregir fondo (azul)
TRAZOS_EJEMPLO = {
    'webcam': (linea(0, (50, 12), (50, 40), 9, 3) + linea(0, (25, 62), (75, 62), 9, 3) +
               linea(0, (40, 30), (60, 30), 7, 3) + linea(0, (44, 25), (56, 25), 3, 1) +
               linea(0, (45, 34), (55, 34), 2, 1) + linea(12, (44, 27), (56, 27), 3, 1)),
    'peatones': linea(0, (60, 47), (60, 63), 3, 3),
}

if MODO_ST == 'interfaz':
    print("Interfaz abierta: Pintá sobre el video y hacé clic en 'Segmentar'")
    interfaz_st(clip, PARAMS_ST)
else:
    TRAZOS = TRAZOS_EJEMPLO.get(VIDEO)
    if TRAZOS is None:
        raise ValueError('Para tu video, usá MODO_ST = "interfaz", o programá TRAZOS manuales.')

    semillas = rasterizar_trazos(TRAZOS, clip.shape[:3])
    t0 = time.time()
    mascaras_st, _ = mincut_st(clip, semillas, **PARAMS_ST)
    RESULTADO_ST.update(mascaras=mascaras_st, semillas=semillas, trazos=TRAZOS)

    tiempo_total = time.time() - t0
    pct_objeto = 100 * mascaras_st.mean()
    print(f'Corte computado en {tiempo_total:.1f} s. El objeto ocupa el {pct_objeto:.1f} % de los píxeles.')



In [ ]:
print("\Grilla de coordenadas (Útil si querés programar trazos a mano en lugar de usar la interfaz):")
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
frames_muestra = (0, T // 2, T - 1)

for ax, t in zip(axs, frames_muestra):
    ax.imshow(clip[t], extent=(0, W, H, 0))
    ax.set_title(f'Frame {t}')
    ax.set_xticks(range(0, W + 1, 10))
    ax.set_yticks(range(0, H + 1, 10))
    ax.grid(alpha=0.5, color='white', linestyle='--')
plt.show()

Cuando estés conforme con la segmentación (después de tocar **Segmentar** en la interfaz), corré esta celda: toma el último resultado y lo deja en `mascaras`.

In [ ]:
if RESULTADO_ST['mascaras'] is None:
    print('Todavía no hay resultado en memoria. Pintá en el widget de arriba y tocá "Segmentar".')
else:
    mascaras = RESULTADO_ST['mascaras']
    pct_objeto = 100 * mascaras.mean()
    print(f'Animación de Salida:')
    print(f'Formato de máscaras: {mascaras.shape}. El objeto ocupa el {pct_objeto:.1f} % del volumen.')


    clip_resaltado = clip * (0.25 + 0.75 * mascaras[..., None])

    # Mostramos el comparativo
    display(mostrar([clip, mascaras, clip_resaltado], ['1. Clip Original', '2. Máscara Booleana', '3. Objeto Aislado']))

## 3.2 Segmentación: Normalized Cut (sin trazos)

Usamos el mismo método espectral de la Parte B, pero aplicado al volumen 3D $(x, y, t)$. Calculamos los autovectores de la ecuación generalizada $(D - W)y = \lambda D y$.

Como el árbol de recursión de la Parte 1 es muy lento para grafos 3D gigantes, implementamos el corte simultáneo en K partes de la Sección 3.4 del paper original:
1.  **K-means (Sobresegmentación):** Se agrupan los píxeles en `K0` segmentos iniciales pequeños, mirando sus valores en el espacio de los primeros autovectores.
2.  **Poda Greedy:** Se fusionan iterativamente de a dos segmentos evaluando cuál fusión minimiza la ecuación general del Ncut, hasta llegar a `K_FINAL` segmentos definitivos.

* Si tu personaje queda fusionado con el fondo: **Subí `K_FINAL`**.
* Si tu personaje queda partido en muchos pedazos: **Bajá `K_FINAL`**.

In [ ]:
#Parámetros del Grafo (Similares a MinCut)
# r: radio espacial | rt: radio temporal
PARAMS_NCUT = dict(r=2, rt=1, sigma_I=0.1, sigma_X=4.0, lam_t=1.0)

#Parámetros de Agrupamiento
K0 = 15 # Segmentos iniciales del K-means (Sobresegmentación)
K_FINAL = 15 # Cantidad de segmentos definitivos que queremos al final

print(f"Calculando Ncut 3D... (Esto puede tardar 30 segundos o más.... (2 o 4 minutos... tal vez) depende de tu video y los autovectores)")

resultado_ncut = ncut_video(clip, PARAMS_NCUT, k_autovectores=10, k0=K0, k_final=K_FINAL)
etiquetas = resultado_ncut['etiquetas']

#Análisis de los segmentos encontrados
print(f"\n Resumen de los {K_FINAL} segmentos definitivos:")
for f in tabla_segmentos(clip, etiquetas):
    pct = 100 * f['tam']
    color = np.round(f['color'], 2)
    print(f" Segmento {f['segmento']}: Ocupa el {pct:5.1f} % del clip | Color medio (RGB): {color}")

# Visualización comparativa
print("\n Animación de Salida:")
display(mostrar(
    [clip, pintar_segmentos(clip, resultado_ncut['sobreseg']), pintar_segmentos(clip, etiquetas)],
    ['1. Clip Original', f'2. K-means ({K0} segs)', f'3. Poda Greedy ({K_FINAL} segs)']
))

## 4. Aplicaciones

1. **Seleccionen UNA aplicación** de la tabla de abajo e implementen el código de sus funciones. Cada aplicación tiene una celda final que renderiza el video con tu efecto.
2. **Generen el video de resultado** (usando un video filmado con el celular, o de internet) y guardenlo usando la función `guardar_video()`.
3. **Respondan las preguntas teóricas** que aparecen al final de la aplicación que elegiste.
4. **Propongan un proyecto más (Sección 5):** Expliquen que efecto nuevo harian, con que método de segmentación (s-t o Ncut), qué funciones implementarías y qué dificultades teóricas/prácticas esperarías. *No hace falta programarlos.*

### Catálogo de Aplicaciones

| Aplicación | Método Recomendado | Descripción |
|---|---|---|
| **Fondo virtual y retrato** | Corte Mínimo s-t | Lo que hacen Zoom, Meet o el modo retrato del celular. |
| **Anonimizar** | Corte Mínimo s-t | Pixelar o censurar a una persona/patente en un video. |
| **Auto-framing (TikTok)** | Normalized Cuts | Recortar un video horizontal a vertical siguiendo al protagonista. |

---

### Funciones de Ayuda
Antes de empezar a programar, pueden usar estas herramientas de NumPy y SciPy.

*   **Filtrado Espacial:** `scipy.ndimage.gaussian_filter()` (ideal para desenfocar fondos en 4.1).
*   **Indexación Booleana:** Si `M` es tu máscara booleana (`True`/`False`), podés modificar solo los píxeles del objeto haciendo `imagen[M] = [255, 0, 0]` o los del fondo haciendo `imagen[~M] = ...`.
*   **Centro de Masa:** `scipy.ndimage.center_of_mass()` (clave para tracking y auto-framing en 4.3 y 4.4).
*   **Slicing 3D:** Recordá que un video de tamaño `(T, H, W, 3)` se puede indexar frame a frame usando un loop `for t in range(T): frame = clip[t]`.

In [ ]:
#Indiquen explícitamente qué método usaste en el Paso 3:
# Opciones: 'mincut' (Corte Mínimo s-t) o 'ncut' (Normalized Cut)
METODO_USADO = 'ncut'

#Si usaron 'ncut', pueden forzar manualmente qué segmento es su protagonista (0, 1, 2...).
#Si dejan None, el código elige automáticamente el de mayor contraste.
ID_SELECCIONADO = None

print(f"Extrayendo máscara generada por: {METODO_USADO.upper()}")


if METODO_USADO == 'mincut':
    if 'RESULTADO_ST' not in locals() or RESULTADO_ST['mascaras'] is None:
        raise ValueError("No hay resultados de MinCut. Volvé a la celda 3.A, pintá tu objeto y tocá 'Segmentar'.")
    mascara_baja_res = RESULTADO_ST['mascaras']

elif METODO_USADO == 'ncut':
    if 'resultado_ncut' not in locals():
        raise ValueError("No hay resultados de Ncut. Volvé y ejecutá la celda 3.B.")

    etiquetas_ncut = resultado_ncut['etiquetas']

    if ID_SELECCIONADO is None:
        #Heuristica
        id_objeto = elegir_por_contraste(clip, etiquetas_ncut)
        print(f"Se seleccionó AUTOMÁTICAMENTE el Segmento {id_objeto} como protagonista.")
    else:
        #Elección manual
        id_objeto = ID_SELECCIONADO
        print(f" Se seleccionó MANUALMENTE el Segmento {id_objeto} como protagonista.")

    #Generamos la máscara booleana aislando ese segmento numérico
    mascara_baja_res = (etiquetas_ncut == id_objeto)

else:
    raise ValueError("La variable METODO_USADO debe ser exactamente 'mincut' o 'ncut'.")

#Limpiamos la máscara (borramos píxeles sueltos y rellenamos huecos)
mascara_limpia = limpiar_mascaras(mascara_baja_res, rellenar=True)

#La escalamos a la resolución del clip HD (Alta definición)
_, H_hd, W_hd, _ = clip_hd.shape
mascaras_objeto = agrandar(mascara_limpia, (H_hd, W_hd))

print("-" * 50)
print(f"Variable `mascaras_objeto` lista para usar en la Aplicación")
print(f"Dimensiones: {mascaras_objeto.shape} (Booleana)")

#Mostramos cómo quedó la silueta final
display(mostrar([mascaras_objeto], ['Máscara Final a resolución HD']))

### Aplicación 0: Efecto "Cartoon" / Caricatura (EJEMPLO RESUELTO)

Para que vean cómo se estructura una aplicación, acá te dejamos un ejemplo resuelto paso a paso.

**Objetivo:** Tomar el resultado de **Normalized Cuts** y promediar el color de los píxeles dentro de cada segmento espacio-temporal. Esto elimina las texturas suaves y deja un efecto visual "plano" parecido a una caricatura.

**Método:** `Ncut` (No supervisado). En este caso, no nos importa aislar un solo "protagonista", sino aplicarle el efecto a *todos* los segmentos que Ncut encontró en el video.

In [ ]:
def aplicar_efecto_cartoon(clip_hd, etiquetas_hd):
    """
    Toma un video y un mapa de etiquetas, y reemplaza el color de todos los píxeles de un mismo segmento por el color promedio de dicho segmento.
    """
    T, H, W, C = clip_hd.shape
    video_salida = np.zeros_like(clip_hd)

    #Obtenemos cuántos segmentos distintos hay
    ids_segmentos = np.unique(etiquetas_hd)

    for id_seg in ids_segmentos:
        #Encontramos la máscara booleana 3D de este segmento específico
        mascara_segmento = (etiquetas_hd == id_seg)

        #Extraemos todos los píxeles del clip original que pertenecen a ese segmento
        pixeles = clip_hd[mascara_segmento]

        #Calculamos el color promedio (media en la dimensión de los píxeles, conservando RGB)
        color_promedio = pixeles.mean(axis=0)

        #Asignamos ese color promedio a todo el segmento en el video de salida
        video_salida[mascara_segmento] = color_promedio

    return video_salida


print("Generando Efecto Cartoon...")

#Como necesitamos TODAS las etiquetas (no solo la del protagonista), traemos el resultado directo de Ncut y lo agrandamos a resolución HD.
if 'resultado_ncut' not in locals():
    raise ValueError("Se debe correr Ncut Primero.")

etiquetas_baja_res = resultado_ncut['etiquetas']
_, H_hd, W_hd, _ = clip_hd.shape

#Escalamos el mapa de etiquetas entero usando interpolación 'nearest' (vecino más cercano) para que los números enteros de las clases (0, 1, 2...) no se rompan
etiquetas_hd = cv2.resize(etiquetas_baja_res.transpose(1, 2, 0).astype(np.uint8), (W_hd, H_hd), interpolation=cv2.INTER_NEAREST).transpose(2, 0, 1)

#Aplicamos el efecto
clip_cartoon = aplicar_efecto_cartoon(clip_hd, etiquetas_hd)

#Mostramos el resultado interactivo
display(mostrar([clip_hd, clip_cartoon], ['1. Video Original', '2. Efecto Cartoon (Promedio Ncut)']))



guardar_video(lado_a_lado(clip_hd, clip_cartoon), "ejemplo_cartoon.mp4", fps=fps_clip)

## TOMEN UNA SOLA Y COMPLETEN

### 4.1 Fondo virtual y modo retrato

Esta aplicación simula el funcionamiento de herramientas como Zoom, Google Meet o el "Modo Retrato" de las cámaras modernas: identificar al sujeto protagonista y reemplazar o alterar todo el fondo que lo rodea.

**El problema del recorte duro:** Si usamos la máscara booleana (`True`/`False`) directamente, el borde entre la persona y el fondo se ve "recortado con tijera", pixelado y poco natural.

**La solución (Alpha Matting):** Para lograr un efecto profesional, convertimos la máscara en una transparencia continua $\alpha \in [0, 1]$. Luego, mezclamos las imágenes usando la ecuación de composición (Alpha Blending):
$$\text{Resultado} = \alpha \cdot \text{Personaje} + (1 - \alpha) \cdot \text{Fondo}$$
donde $\alpha$ es la máscara suavizada.

In [ ]:
def suavizar_mascara(mascaras, sigma=1.5):
    '''
    Toma máscaras booleanas y devuelve canales alfa continuos [0, 1] suavizando los bordes.

    Parámetros:
      mascaras: arreglo booleano de forma (T, H, W).
      sigma: nivel de desenfoque del borde.

    Devuelve:
      alfa: arreglo float32 de forma (T, H, W) con valores entre 0.0 y 1.0.
    '''
    T, H, W = mascaras.shape
    alfa = np.zeros((T, H, W), dtype=np.float32)

    # PISTA: Iterá frame por frame
    # Convertí cada frame booleano a float usando .astype(np.float32).
    # Aplicale cv2.GaussianBlur(frame_float, (0, 0), sigma) y guardalo en alfa[t].

    pass

    return alfa


def reemplazar_fondo(clip, mascaras, fondo_nuevo, sigma_borde=0):
    '''
    Reemplaza el fondo de un video por una imagen estática.

    Parámetros:
      clip: video original (T, H, W, 3).
      mascaras: máscaras booleanas del protagonista (T, H, W).
      fondo_nuevo: imagen RGB en rango [0, 1]. Puede no coincidir en resolución.
      sigma_borde: si es > 0, suaviza la transición. Si es 0, recorte duro.

    Devuelve:
      resultado: video compuesto (T, H, W, 3).
    '''
    T, H, W, C = clip.shape

    #PISTA 1: Redimensioná 'fondo_nuevo' para que tenga tamaño (W, H) usando cv2.resize().

    #PISTA 2: Si sigma_borde > 0, calculá 'alfa' usando tu función suavizar_mascara().
    #OJO: Para multiplicar, 'alfa' debe tener forma (T, H, W, 1). Podés hacer alfa = alfa[..., None].
    #Aplicá la fórmula de Alpha Blending.

    #PISTA 3: Si sigma_borde es 0, usá np.where() o indexación booleana


    pass


def desenfocar_fondo(clip, mascaras, sigma=5, sigma_borde=1.5):
    '''
    Simula el "Modo Retrato", difuminando el fondo original del video.

    Parámetros:
      clip: video original (T, H, W, 3).
      mascaras: máscaras booleanas (T, H, W).
      sigma: fuerza del desenfoque (blur) aplicado al fondo.
      sigma_borde: desenfoque del canal alfa para la transición sujeto/fondo.

    Devuelve:
      retrato: video compuesto (T, H, W, 3).
    '''
    #PISTA 1: Vas a necesitar un clip auxiliar (ej: 'fondo_borroso') donde cada frame del clip original esté procesado con cv2.GaussianBlur(..., sigma).

    #PISTA 2: Obtené el 'alfa' con suavizar_mascara(mascaras, sigma_borde) y expandile la dimensión de los canales (alfa[..., None]).

    #PISTA 3: Mezclá el clip original (persona) y el fondo_borroso usando la fórmula de Alpha Blending.

    pass

In [ ]:
if 'mascaras_objeto' not in locals():
    raise ValueError("Corré primero la celda de 'Extracción de la Máscara' que está justo arriba.")
m_hd = mascaras_objeto

#Creamos un fondo falso (cielo y arena) para probar. (Podés cargar una imagen tuya haciendo: fondo = np.asarray(Image.open('playa.jpg')) / 255.0 )
fondo_playa = np.zeros((10, 10, 3), np.float32)
fondo_playa[:6] = (0.45, 0.7, 1.0) # Cielo (Celeste)
fondo_playa[6:] = (0.95, 0.85, 0.55) # Arena (Amarillento)

print("Procesando efectos de Fondo Virtual...")

# Ejecutamos tus funciones
duro = reemplazar_fondo(clip_hd, m_hd, fondo_playa, sigma_borde=0)
suave = reemplazar_fondo(clip_hd, m_hd, fondo_playa, sigma_borde=3)
retrato = desenfocar_fondo(clip_hd, m_hd, sigma=8, sigma_borde=3)

# Mostramos los 4 paneles comparativos
display(mostrar(
    [clip_hd, duro, suave, retrato],
    ['1. Original', '2. Borde Duro (feo)', '3. Borde Suave (Alpha)', '4. Modo Retrato']
))

guardar_video(lado_a_lado(clip_hd, suave, retrato), 'fondo_virtual.mp4', fps=fps_clip)

**Preguntas (4.1):**
1. Compará el borde duro con el suave. ¿Por qué el suave se ve más natural? ¿Qué pasa si `sigma_borde` es muy grande?
2. ¿Dónde falla la segmentación? Pensá en el pelo, las manos que se mueven rápido, o ropa del mismo color que la pared.
   ¿Cuáles de esos errores se arreglan con más trazos y cuáles no?
3. Las apps de videollamada hacen esto en tiempo real, a 30 fps. ¿Podrían usar este método tal cual? ¿Qué parte es la
   cara (en tiempo de cómputo) y qué parte necesita a una persona?

### 4.2 Anonimizar (Pixelado por privacidad)

Antes de publicar un video en internet (noticieros, documentales o Google Street View), muchas veces es obligatorio por ley ocultar la identidad de personas que no dieron su consentimiento o tapar patentes de autos. La segmentación que hicieron en el paso anterior nos dice exactamente **dónde** aplicar el filtro.

**¿Cómo funciona el pixelado?**
La forma más eficiente de pixelar un frame no es usar bucles `for` complejos, sino usar matemática de escalado de imágenes:
1. Achicamos la imagen drásticamente (ej: a una décima parte) usando interpolación de área (`cv2.INTER_AREA`). Esto promedia los colores.
2. La volvemos a agrandar a su tamaño original usando interpolación de vecino más cercano (`cv2.INTER_NEAREST`). Esto hace que cada píxel promediado se dibuje como un "cuadrado gigante".

In [ ]:
def pixelar(frame, mascara, bloque=8):
    '''
    Pixela las regiones indicadas por la máscara.

    Parámetros:
      frame: imagen original de un instante de tiempo (H, W, 3).
      mascara: máscara booleana de ese frame (H, W). True = pixelar, False = original.
      bloque: tamaño del "cuadrado" de pixelado (ej: 8x8 píxeles).

    Devuelve:
      frame_modificado: imagen (H, W, 3) con el sujeto pixelado.
    '''
    H, W, C = frame.shape

    #PISTA 1: Creá la versión pixelada de todo el frame.
    #Primero, achicá el frame:
    #pequeno = cv2.resize(frame, (W // bloque, H // bloque), interpolation=cv2.INTER_AREA)
    #Segundo, volvelo a agrandar al tamaño (W, H):
    #frame_pixelado = cv2.resize(pequeno, (W, H), interpolation=cv2.INTER_NEAREST)

    # PISTA 2: Usá np.where o indexación booleana para combinar el 'frame_pixelado' y el 'frame' original guiándote por la 'mascara'.
    #Recorda que si usás np.where, la máscara booleana debe tener forma (H, W, 1).

    pass


def anonimizar_clip(clip, mascaras, bloque=8, margen=4):
    '''
    Aplica el efecto de pixelado a todo el video. Para mayor seguridad, expande la máscara unos píxeles para evitar que se vean contornos.

    Parámetros:
      clip: video original (T, H, W, 3).
      mascaras: máscaras booleanas del protagonista (T, H, W).
      bloque: tamaño del cuadrado de pixelado.
      margen: cuántos píxeles agrandar la máscara (dilatación).

    Devuelve:
      clip_anonimo: video compuesto (T, H, W, 3).
    '''
    T, H, W, C = clip.shape
    clip_anonimo = np.zeros_like(clip)

    #Creamos el kernel estructural para la dilatación
    nucleo = np.ones((2 * margen + 1, 2 * margen + 1), dtype=np.uint8)

    for t in range(T):
        #PISTA 1: Dilatá (expandí) la máscara de este frame para cubrir más terreno. Recordá que cv2.dilate pide tipo uint8.
        #m_expandida = cv2.dilate(mascaras[t].astype(np.uint8), nucleo).astype(bool)

        #PISTA 2: Llamá a tu función pixelar() pasándole el frame original, la m_expandida y el parámetro de bloque. Guardá el resultado en clip_anonimo[t].

        pass

    return clip_anonimo

In [ ]:
if 'mascaras_objeto' not in locals():
    raise ValueError("Corré primero la celda de 'Extracción de la Máscara' de la Sección 4.")
m_hd = mascaras_objeto

print("Aplicando filtro de anonimización...")

#Ejecutamos tu función con dos configuraciones para comparar
sin_margen = anonimizar_clip(clip_hd, m_hd, bloque=12, margen=0)
con_margen = anonimizar_clip(clip_hd, m_hd, bloque=12, margen=8)

#Mostramos comparativa
display(mostrar(
    [clip_hd, sin_margen, con_margen],
    ['1. Original', '2. Sin Margen (Riesgo de fuga)', '3. Con Margen (Seguro)']
))

#(Opcional) Si querés guardar el resultado final en un archivo
guardar_video(lado_a_lado(clip_hd, sin_margen, con_margen), 'anonimizado.mp4', fps=fps_clip)

**Preguntas (4.2):**
1. ¿Por qué hace falta el `margen`? Mirá los bordes del objeto en la versión sin margen.
2. Si en **un solo frame** la segmentación pierde una parte de la persona, ¿qué pasa con la anonimización? ¿Por qué en
   esta aplicación un error chico es grave? ¿Qué cambiarías para estar más seguro (pensá en `margen`, en `lam_t`, o en
   pixelar de más)?
3. ¿Se podría anonimizar sin que nadie pinte, con Ncut? ¿Qué le faltaría a Ncut para saber **qué** ocultar?

### 4.3 Reencuadre automático a formato vertical (TikTok / Reels)

Para subir un video horizontal tradicional a las redes sociales, hay que recortarlo al formato 9:16 (vertical). Pero no podés recortar siempre el centro, porque el sujeto podría moverse hacia los lados y salirse del encuadre.

El *Auto Reframe* (que traen editores como Premiere Pro) resuelve esto siguiendo automáticamente al protagonista. Acá **nadie pinta nada**: usaremos **Normalized Cuts (Ncut)** el algoritmo encuentre todos los segmentos del video por sí sola, y luego tu código decidirá lógicamente cuál de esos K segmentos es el protagonista.

**¿Cómo deducimos cuál es el protagonista sin intervención humana?**
Una heurística fotográfica simple: el fondo (cielo, pared, piso) casi siempre toca los bordes del video, pero el protagonista (una persona, un auto) suele estar más hacia el centro.
Regla: Buscá el segmento que, ocupando al menos un `tam_min` del video, tenga la **menor fracción de sus píxeles pegados a los bordes de la imagen**.

In [ ]:
import scipy.ndimage
def centroide(mascara):
    '''Calcula el centro de masa de una máscara 2D.'''
    if not mascara.any():
        return np.array([np.nan, np.nan])
    # scipy.ndimage.center_of_mass devuelve (fila, columna)
    return np.array(scipy.ndimage.center_of_mass(mascara))

def trayectoria(mascaras):
    '''Calcula el recorrido [fila, columna] del objeto a lo largo del clip.'''
    T = len(mascaras)
    tray = np.zeros((T, 2))
    for t in range(T):
        tray[t] = centroide(mascaras[t])
    return tray

def segmento_principal(etiquetas, tam_min=0.03):
    '''
    Heurística para encontrar al protagonista de forma no supervisada.

    Parámetros:
      etiquetas: (T, H, W) mapa de segmentos [0, 1, 2... K] de Ncut.
      tam_min: el segmento debe ocupar al menos esta fracción (3%) del clip.
    Devuelve:
      mejor_segmento: el número entero del segmento protagonista.
    '''
    T, H, W = etiquetas.shape

    # 1. Armamos una máscara booleana 2D indicando qué píxeles son "borde de imagen"
    borde = np.zeros((H, W), dtype=bool)
    borde[0, :] = borde[-1, :] = borde[:, 0] = borde[:, -1] = True

    mejor_c, mejor_frac = None, np.inf

    for c in np.unique(etiquetas):
        # PISTA 1: Generá la máscara booleana 3D 'm' para el segmento 'c'.
        # PISTA 2: Si el promedio m.mean() es menor a 'tam_min', descartalo (continue).
        # PISTA 3: Calculá cuántos píxeles de 'm' caen justo sobre los píxeles de 'borde'.
        #          frac_borde = m[:, borde].sum() / m.sum()
        # PISTA 4: Si esa fracción es menor a 'mejor_frac', actualizá 'mejor_c' y 'mejor_frac'.

        # --- TU CÓDIGO AQUÍ ---
        pass
        # ----------------------

    return mejor_c


def suavizar_trayectoria(centros, ventana=5):
    '''
    Suaviza el movimiento de cámara para que el recorte no tiemble.

    Parámetros:
      centros: (T, 2) posiciones del centroide. Puede tener NaN (sujeto perdido).
      ventana: cantidad de frames para el promedio móvil.
    Devuelve:
      trayectoria_suave: (T, 2).
    '''
    c = centros.copy()

    # 1. Rellenar agujeros (NaNs)
    for t in range(len(c)):
        if np.isnan(c[t]).any():
            c[t] = c[t - 1] if t > 0 else np.nanmean(centros, axis=0)

    # 2. Promedio Móvil (Filtro pasa-bajos 1D)
    # PISTA: Rellená los bordes de 'c' temporalmente para no perder frames al aplicar
    # la convolución. Usá np.pad(c, ((ventana // 2, ventana // 2), (0, 0)), mode='edge').
    # Luego pasá un np.convolve a cada columna.

    # --- TU CÓDIGO AQUÍ ---
    pass
    # ----------------------


def recortar_vertical(clip_hd, centros, escala, proporcion=9/16):
    '''
    Hace el paneo y recorte automático del video.

    Parámetros:
      clip_hd: (T, H, W, 3) video de alta calidad.
      centros: (T, 2) centros de masa calculados sobre la baja calidad.
      escala: multiplicador para pasar de pixeles chicos a pixeles HD.
      proporcion: relación de aspecto (9 de ancho por 16 de alto).
    Devuelve:
      salida: (T, H_hd, W_recortado, 3).
    '''
    T, H, W = clip_hd.shape[:3]

    # Calculamos el ancho ideal para mantener la altura completa de HD
    ancho = min(W, int(round(H * proporcion)))
    salida = []

    for t in range(T):
        # PISTA: Encontrá la columna central del protagonista (centros[t, 1] * escala).
        # Restale la mitad del 'ancho' de ventana que calculaste arriba para obtener la columna de inicio (x0).
        # OJO: Asegurate de usar np.clip para que x0 nunca sea menor a 0 ni mayor a (W - ancho).
        # Finalmente, extraé clip_hd[t, :, x0 : x0 + ancho] y hacé append a la salida.

        # --- TU CÓDIGO AQUÍ ---
        pass
        # ----------------------

    return np.stack(salida)

In [ ]:
if 'resultado_ncut' not in locals():
    raise ValueError("Para Auto-framing se exige usar Ncut (Celda 3.B). Corré esa celda primero.")

etiquetas_bajas = resultado_ncut['etiquetas']
T_limite = min(len(clip_hd), len(etiquetas_bajas))

# BÚSQUEDA DEL PROTAGONISTA
print("Buscando sujeto principal (Heurística de Bordes)...")
protagonista = segmento_principal(etiquetas_bajas[:T_limite])
m_prot = limpiar_mascaras(etiquetas_bajas[:T_limite] == protagonista)

pct_clip = 100 * m_prot.mean()
print(f'Protagonista encontrado: Segmento {protagonista} (Ocupa {pct_clip:.1f}% del clip).')

#CALCULAR TRAYECTORIA Y SUAVIZAR (STEADY-CAM)
print("Calculando paneo de cámara (Steady-Cam)...")
tray_bruta = trayectoria(m_prot)  # Asume que ya hiciste trayectoria() en el ej 4.3!
centros = suavizar_trayectoria(tray_bruta, ventana=5)

#RECORTAR A 9:16 (TIKTOK)
print("Recortando y renderizando HD...")
escala_hd = clip_hd.shape[2] / clip.shape[2]
vertical = recortar_vertical(clip_hd[:T_limite], centros, escala=escala_hd)

#Dibujamos el rectángulo de encuadre sobre el video original
marcado = clip_hd[:T_limite].copy()
ancho_v = vertical.shape[2]
H_marcado = marcado.shape[1]
W_marcado = marcado.shape[2]

for t in range(len(marcado)):
    x0 = int(np.clip(round(centros[t, 1] * escala_hd - ancho_v / 2), 0, W_marcado - ancho_v))
    #Dibujamos un rectángulo verde lima
    cv2.rectangle(marcado[t], (x0, 0), (x0 + ancho_v - 1, H_marcado - 1), (0, 255, 0), 4)

display(mostrar(
    [marcado, m_prot, vertical],
    ['1. Tracking de Encuadre', '2. Máscara del Protagonista', '3. Video Vertical 9:16']
))

# (Opcional) Guardar
guardar_video(lado_a_lado(marcado, m_prot, vertical), 'auto_tiktok.mp4', fps=fps_clip)

**Preguntas (4.4):**
1. Probá con `ventana=1` (sin suavizar). ¿Qué le pasa a la "cámara"? ¿Por qué el suavizado es parte del producto y no
   un detalle?
2. ¿Qué pasa si hay dos personas, o si el protagonista sale del cuadro? ¿Cómo cambiarías `segmento_principal`?
3. ¿Por qué acá conviene Ncut y no s-t? Pensá en una plataforma que tiene que reencuadrar mil videos por hora.

## 5. Dos proyectos más (sin programar)

Proponé **dos proyectos** que se podrían construir con estas herramientas de segmentacion (puede ser en video o iamgen fija), puedens er utiles o absurdas.

### Proyecto 1: *(nombre)*

- **Qué hace**
- **Para quien o que institucion/rubro esta dirigido**
- **Que Método (s-t o Ncut) y por qué:**
- **Qué funciones implementarías** (nombre, entrada, salida):
- **Qué dificultades esperás** (del método, del video o del problema) **y cómo las enfrentarías (si se les ocurre algo):**

### Proyecto 2: *(nombre)*

- **Qué hace**
- **Para quien o que institucion/rubro esta dirigido**
- **Que Método (s-t o Ncut) y por qué:**
- **Qué funciones implementarías** (nombre, entrada, salida):
- **Qué dificultades esperás** (del método, del video o del problema) **y cómo las enfrentarías (si se les ocurre algo):**